<a href="https://colab.research.google.com/github/san-258/Pullback-to-21-ema-scan-for-D-1Hr-15min/blob/Only-Daily/Pullback%20to%2021%20ema%2C%20scan%20for%20D/1Hr/%2015min.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Google Colab Stock Scanner - daily Technical Analysis
# Run this cell to install required packages
!pip install yfinance pandas numpy schedule plotly

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time
import json
import warnings
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, clear_output, HTML
import threading
from google.colab import files
warnings.filterwarnings('ignore')

class ColabTechnicalScanner:
    def __init__(self, tickers):
        self.tickers = list(set(tickers))  # Remove duplicates
        self.results = []
        self.previous_results = {}
        self.alerts = []
        self.scan_history = []
        self.is_running = False

    def calculate_ema(self, prices, period):
        """Calculate Exponential Moving Average"""
        return prices.ewm(span=period).mean()

    def calculate_rsi(self, prices, period=14):
        """Calculate RSI"""
        delta = prices.diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        return rsi

    def is_hammer(self, open_price, high, low, close):
        """Identify Hammer candlestick pattern"""
        body = abs(close - open_price)
        upper_shadow = high - max(open_price, close)
        lower_shadow = min(open_price, close) - low
        total_range = high - low

        if total_range == 0:
            return False

        return (lower_shadow > 2 * body and
                upper_shadow < body and
                body > 0 and
                lower_shadow > 0.6 * total_range)  # Strong hammer criteria

    def is_inverted_hammer(self, open_price, high, low, close):
        """Identify Inverted Hammer pattern"""
        body = abs(close - open_price)
        upper_shadow = high - max(open_price, close)
        lower_shadow = min(open_price, close) - low
        total_range = high - low

        if total_range == 0:
            return False

        return (upper_shadow > 2 * body and
                lower_shadow < body and
                body > 0 and
                upper_shadow > 0.6 * total_range)

    def is_bullish_engulfing(self, prev_open, prev_close, curr_open, curr_close):
        """Identify Bullish Engulfing pattern"""
        prev_bearish = prev_close < prev_open
        curr_bullish = curr_close > curr_open
        engulfing = curr_open <= prev_close and curr_close > prev_open
        significant_body = abs(curr_close - curr_open) > abs(prev_close - prev_open) * 1.1

        return prev_bearish and curr_bullish and engulfing and significant_body

    def is_piercing_pattern(self, prev_open, prev_high, prev_low, prev_close,
                           curr_open, curr_high, curr_low, curr_close):
        """Identify Piercing Line pattern"""
        prev_bearish = prev_close < prev_open
        curr_bullish = curr_close > curr_open
        gap_down = curr_open < prev_low
        penetration = curr_close > (prev_open + prev_close) / 2

        return prev_bearish and curr_bullish and gap_down and penetration

    def is_morning_star(self, candles):
        """Identify Morning Star pattern (3-candle pattern)"""
        if len(candles) < 3:
            return False

        first, second, third = candles[-3], candles[-2], candles[-1]

        # First candle: Bearish
        first_bearish = first['Close'] < first['Open']
        first_body = abs(first['Close'] - first['Open'])

        # Second candle: Small body (star)
        second_body = abs(second['Close'] - second['Open'])
        second_small = second_body < first_body * 0.3
        gap_down = second['High'] < first['Close']

        # Third candle: Bullish
        third_bullish = third['Close'] > third['Open']
        third_body = abs(third['Close'] - third['Open'])
        penetration = third['Close'] > (first['Open'] + first['Close']) / 2

        return (first_bearish and second_small and gap_down and
                third_bullish and penetration and third_body > first_body * 0.6)

    def is_bullish_harami(self, prev_open, prev_high, prev_low, prev_close,
                         curr_open, curr_high, curr_low, curr_close):
        """Identify Bullish Harami pattern"""
        prev_bearish = prev_close < prev_open
        curr_bullish = curr_close > curr_open

        # Current candle contained within previous candle's body
        contained = (curr_open > prev_close and curr_open < prev_open and
                    curr_close > prev_close and curr_close < prev_open)

        prev_body = abs(prev_close - prev_open)
        curr_body = abs(curr_close - curr_open)
        significant_prev = prev_body > (prev_high - prev_low) * 0.6

        return prev_bearish and curr_bullish and contained and significant_prev

    def is_dragonfly_doji(self, open_price, high, low, close):
        """Identify Dragonfly Doji pattern"""
        body = abs(close - open_price)
        upper_shadow = high - max(open_price, close)
        lower_shadow = min(open_price, close) - low
        total_range = high - low

        if total_range == 0:
            return False

        return (body <= total_range * 0.1 and
                upper_shadow <= total_range * 0.1 and
                lower_shadow >= total_range * 0.7)

    def is_three_white_soldiers(self, candles):
        """Identify Three White Soldiers pattern"""
        if len(candles) < 3:
            return False

        last_three = candles[-3:]

        for candle in last_three:
            # All must be bullish
            if candle['Close'] <= candle['Open']:
                return False

            # Each should have reasonable body size
            body = abs(candle['Close'] - candle['Open'])
            total_range = candle['High'] - candle['Low']
            if body < total_range * 0.6:  # Strong bullish bodies
                return False

        # Each close should be higher than previous
        for i in range(1, 3):
            if last_three[i]['Close'] <= last_three[i-1]['Close']:
                return False

        # Each open should be within previous candle's body
        for i in range(1, 3):
            prev_body_mid = (last_three[i-1]['Open'] + last_three[i-1]['Close']) / 2
            if last_three[i]['Open'] < prev_body_mid:
                return False

        return True

    def is_abandoned_baby_bullish(self, candles):
        """Identify Bullish Abandoned Baby pattern"""
        if len(candles) < 3:
            return False

        first, second, third = candles[-3], candles[-2], candles[-1]

        # First: Strong bearish
        first_bearish = first['Close'] < first['Open']
        first_body = abs(first['Close'] - first['Open'])

        # Second: Doji with gaps
        second_doji = abs(second['Close'] - second['Open']) < (second['High'] - second['Low']) * 0.1
        gap_down = second['High'] < first['Low']
        gap_up = third['Low'] > second['High']

        # Third: Strong bullish
        third_bullish = third['Close'] > third['Open']
        third_body = abs(third['Close'] - third['Open'])

        return (first_bearish and second_doji and gap_down and gap_up and
                third_bullish and first_body > 0 and third_body > 0)

    def is_doji(self, open_price, close, high, low):
        """Identify Doji pattern (enhanced)"""
        body = abs(close - open_price)
        total_range = high - low

        if total_range == 0:
            return False

        return body <= (total_range * 0.1)

    def get_market_data(self, ticker):
        """Get daily market data for candlestick pattern analysis"""
        try:
            stock = yf.Ticker(ticker)

            # Get daily data (1 year for comprehensive analysis)
            daily = stock.history(period="1y", interval="1d")

            return daily
        except Exception as e:
            return None

    def analyze_stock(self, ticker):
        """Analyze individual stock using daily timeframe"""
        try:
            daily = self.get_market_data(ticker)

            if daily is None or len(daily) < 200:
                return None

            # Calculate daily EMAs and indicators
            daily['EMA_21'] = self.calculate_ema(daily['Close'], 21)
            daily['EMA_50'] = self.calculate_ema(daily['Close'], 50)
            daily['EMA_200'] = self.calculate_ema(daily['Close'], 200)
            daily['RSI'] = self.calculate_rsi(daily['Close'], 14)
            daily['Volume_MA'] = daily['Volume'].rolling(window=20).mean()

            # Current values
            current = daily.iloc[-1]
            prev = daily.iloc[-2] if len(daily) > 1 else current

            current_price = current['Close']

            # Trend confirmation
            trend_confirmed = (current['EMA_50'] > current['EMA_200'] and
                             current_price > current['EMA_21'])

            # Setup criteria (daily timeframe)
            price_near_ema21 = abs(current_price - current['EMA_21']) / current['EMA_21'] <= 0.02
            rsi_in_range = 30 <= current['RSI'] <= 40  # Original RSI range
            volume_above_avg = current['Volume'] >= (1.5 * current['Volume_MA'])  # Original volume threshold

            # Enhanced reversal pattern detection using daily candles
            candles_data = []
            if len(daily) >= 3:
                for i in range(-3, 0):
                    candles_data.append({
                        'Open': daily.iloc[i]['Open'],
                        'High': daily.iloc[i]['High'],
                        'Low': daily.iloc[i]['Low'],
                        'Close': daily.iloc[i]['Close']
                    })

            # Single candle patterns
            hammer = self.is_hammer(current['Open'], current['High'], current['Low'], current['Close'])
            inverted_hammer = self.is_inverted_hammer(current['Open'], current['High'], current['Low'], current['Close'])
            dragonfly_doji = self.is_dragonfly_doji(current['Open'], current['High'], current['Low'], current['Close'])
            regular_doji = self.is_doji(current['Open'], current['Close'], current['High'], current['Low'])

            # Two candle patterns
            bullish_engulfing = self.is_bullish_engulfing(prev['Open'], prev['Close'],
                                                         current['Open'], current['Close'])
            piercing_pattern = self.is_piercing_pattern(prev['Open'], prev['High'], prev['Low'], prev['Close'],
                                                       current['Open'], current['High'], current['Low'], current['Close'])
            bullish_harami = self.is_bullish_harami(prev['Open'], prev['High'], prev['Low'], prev['Close'],
                                                   current['Open'], current['High'], current['Low'], current['Close'])

            # Three candle patterns
            morning_star = self.is_morning_star(candles_data) if len(candles_data) >= 3 else False
            three_white_soldiers = self.is_three_white_soldiers(candles_data) if len(candles_data) >= 3 else False
            abandoned_baby_bullish = self.is_abandoned_baby_bullish(candles_data) if len(candles_data) >= 3 else False

            # Aggregate reversal signal
            powerful_patterns = [bullish_engulfing, morning_star, three_white_soldiers, abandoned_baby_bullish]
            moderate_patterns = [hammer, piercing_pattern, bullish_harami, inverted_hammer]
            weak_patterns = [dragonfly_doji, regular_doji]

            # Pattern strength scoring
            pattern_strength = 0
            if any(powerful_patterns):
                pattern_strength = 3  # Very strong
            elif any(moderate_patterns):
                pattern_strength = 2  # Strong
            elif any(weak_patterns) and current['RSI'] < 35:
                pattern_strength = 1  # Moderate (only if oversold)

            reversal_candle = pattern_strength >= 2  # Require at least strong patterns

            # Identify specific patterns found
            patterns_found = []
            if bullish_engulfing: patterns_found.append("Bullish Engulfing")
            if morning_star: patterns_found.append("Morning Star")
            if three_white_soldiers: patterns_found.append("Three White Soldiers")
            if abandoned_baby_bullish: patterns_found.append("Abandoned Baby")
            if hammer: patterns_found.append("Hammer")
            if piercing_pattern: patterns_found.append("Piercing Pattern")
            if bullish_harami: patterns_found.append("Bullish Harami")
            if inverted_hammer: patterns_found.append("Inverted Hammer")
            if dragonfly_doji: patterns_found.append("Dragonfly Doji")
            if regular_doji and current['RSI'] < 35: patterns_found.append("Doji (Oversold)")

            # Setup A qualification
            setup_a = (price_near_ema21 and rsi_in_range and
                      reversal_candle and volume_above_avg and trend_confirmed)

            # Daily momentum score
            momentum_score = self.calculate_daily_momentum(current, daily)

            return {
                'ticker': ticker,
                'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                'price': round(current_price, 2),
                'ema_21': round(current['EMA_21'], 2),
                'ema_50': round(current['EMA_50'], 2),
                'ema_200': round(current['EMA_200'], 2),
                'rsi': round(current['RSI'], 2),
                'volume': int(current['Volume']),
                'volume_avg': int(current['Volume_MA']) if not pd.isna(current['Volume_MA']) else 0,
                'trend_confirmed': trend_confirmed,
                'price_near_ema21': price_near_ema21,
                'rsi_in_range': rsi_in_range,
                'volume_above_avg': volume_above_avg,
                'reversal_candle': reversal_candle,
                'pattern_strength': pattern_strength,
                'patterns_found': patterns_found,
                'setup_a': setup_a,
                'momentum': momentum_score,
                'ema21_distance_pct': round(((current_price - current['EMA_21']) / current['EMA_21']) * 100, 2),
                # Individual pattern flags for detailed analysis
                'bullish_engulfing': bullish_engulfing,
                'morning_star': morning_star,
                'three_white_soldiers': three_white_soldiers,
                'abandoned_baby': abandoned_baby_bullish,
                'hammer': hammer,
                'piercing_pattern': piercing_pattern,
                'bullish_harami': bullish_harami,
                'inverted_hammer': inverted_hammer,
                'dragonfly_doji': dragonfly_doji
            }

        except Exception as e:
            return None

    def calculate_daily_momentum(self, current, daily_data):
        """Calculate momentum score for daily timeframe"""
        try:
            rsi = current['RSI']
            vol_ratio = current['Volume'] / current['Volume_MA'] if current['Volume_MA'] > 0 else 1

            # Price change over last 5 days
            if len(daily_data) >= 5:
                five_day_change = (current['Close'] - daily_data.iloc[-5]['Close']) / daily_data.iloc[-5]['Close']
            else:
                five_day_change = 0

            # Price change over last 10 days
            if len(daily_data) >= 10:
                ten_day_change = (current['Close'] - daily_data.iloc[-10]['Close']) / daily_data.iloc[-10]['Close']
            else:
                ten_day_change = 0

            # Relative strength vs EMA
            ema_strength = (current['Close'] - current['EMA_21']) / current['EMA_21'] if current['EMA_21'] > 0 else 0

            momentum = (
                (50 - abs(rsi - 35)) * 0.3 +  # RSI near 35 is ideal for pullback
                min(vol_ratio * 20, 30) * 0.25 +  # Volume boost
                max(five_day_change * 500, -15) * 0.25 +  # 5-day momentum
                max(ten_day_change * 300, -10) * 0.1 +  # 10-day momentum
                max(ema_strength * 100, -10) * 0.1  # EMA relationship
            )

            return round(max(0, min(100, momentum)), 1)
        except:
            return 0

    def is_market_hours(self):
        """Check if market is open"""
        now = datetime.now()
        if now.weekday() >= 5:  # Weekend
            return False

        hour = now.hour
        return 9 <= hour <= 16  # Simplified market hours check

    def run_scan(self):
        """Run a single daily scan"""
        print(f"🔍 Starting daily scan at {datetime.now().strftime('%H:%M:%S')}")
        print(f"📊 Analyzing {len(self.tickers)} tickers with daily candlestick patterns...")

        self.results = []
        scan_start = time.time()

        for i, ticker in enumerate(self.tickers):
            if i % 10 == 0:  # Progress every 10 stocks
                print(f"   Progress: {i+1}/{len(self.tickers)}")

            result = self.analyze_stock(ticker)
            if result:
                self.results.append(result)

        scan_time = time.time() - scan_start
        print(f"✅ Daily scan completed in {scan_time:.1f} seconds")

        # Store scan in history
        self.scan_history.append({
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'results_count': len(self.results),
            'setup_a_count': len([r for r in self.results if r['setup_a']])
        })

        return self.results

    def display_results(self):
        """Display results in Colab-friendly format"""
        if not self.results:
            print("❌ No results to display")
            return

        df = pd.DataFrame(self.results)

        # Setup A stocks
        setup_a = df[df['setup_a'] == True].sort_values('momentum', ascending=False)

        print("🎯 SETUP A QUALIFIED STOCKS")
        print("=" * 80)
        if len(setup_a) > 0:
            for _, stock in setup_a.iterrows():
                patterns_str = ", ".join(stock['patterns_found']) if stock['patterns_found'] else "Other"
                strength_emoji = "🔥" if stock['pattern_strength'] == 3 else "⚡" if stock['pattern_strength'] == 2 else "📈"

                print(f"{strength_emoji} {stock['ticker']} - ${stock['price']}")
                print(f"   RSI: {stock['rsi']} | Momentum: {stock['momentum']} | EMA21 Dist: {stock['ema21_distance_pct']}%")
                print(f"   Volume: {stock['volume']:,} ({stock['volume']/stock['volume_avg']:.1f}x avg)")
                print(f"   🎯 Pattern: {patterns_str} (Strength: {stock['pattern_strength']}/3)")
                print()
        else:
            print("No stocks currently meet all Setup A criteria")

        # High momentum stocks
        high_momentum = df[df['momentum'] > 50].sort_values('momentum', ascending=False)

        print("🚀 HIGH MOMENTUM STOCKS (Top 15)")
        print("=" * 60)
        for _, stock in high_momentum.head(15).iterrows():
            status = "🎯" if stock['setup_a'] else "📈"
            print(f"{status} {stock['ticker']}: ${stock['price']} | RSI: {stock['rsi']} | Momentum: {stock['momentum']}")

        # Powerful pattern alerts
        powerful_patterns = df[(df['pattern_strength'] >= 3) & (df['trend_confirmed'] == True)]
        if len(powerful_patterns) > 0:
            print(f"\n🔥 POWERFUL REVERSAL PATTERNS ({len(powerful_patterns)})")
            print("=" * 60)
            for _, stock in powerful_patterns.iterrows():
                patterns_str = ", ".join(stock['patterns_found'])
                print(f"🔥 {stock['ticker']}: ${stock['price']} | RSI: {stock['rsi']} | Pattern: {patterns_str}")

        # Pattern breakdown
        pattern_summary = {}
        for _, stock in df.iterrows():
            for pattern in stock['patterns_found']:
                if pattern not in pattern_summary:
                    pattern_summary[pattern] = 0
                pattern_summary[pattern] += 1

        if pattern_summary:
            print(f"\n📊 PATTERN BREAKDOWN:")
            print("=" * 40)
            for pattern, count in sorted(pattern_summary.items(), key=lambda x: x[1], reverse=True):
                print(f"   {pattern}: {count} stocks")

        # Watch zone stocks
        watch_zone = df[(df['rsi_in_range']) & (df['price_near_ema21']) &
                       (df['trend_confirmed']) & (~df['setup_a'])]

        if len(watch_zone) > 0:
            print(f"\n👀 WATCH ZONE STOCKS ({len(watch_zone)})")
            print("=" * 50)
            for _, stock in watch_zone.iterrows():
                patterns_str = ", ".join(stock['patterns_found']) if stock['patterns_found'] else "Waiting for signal"
                print(f"⏰ {stock['ticker']}: ${stock['price']} | RSI: {stock['rsi']} | {patterns_str}")

        # Summary
        print(f"\n📈 SUMMARY")
        print("=" * 30)
        print(f"Total analyzed: {len(df)}")
        print(f"Setup A qualified: {len(setup_a)}")
        print(f"Powerful patterns (3/3): {len(df[df['pattern_strength'] >= 3])}")
        print(f"Strong patterns (2/3): {len(df[df['pattern_strength'] >= 2])}")
        print(f"Trend confirmed: {len(df[df['trend_confirmed']])}")
        print(f"High momentum (>50): {len(high_momentum)}")
        print(f"Average RSI: {df['rsi'].mean():.1f}")

        return df

    def create_visualization(self, df):
        """Create interactive charts"""
        if df is None or len(df) == 0:
            return

        # RSI Distribution
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=('RSI Distribution', 'Momentum vs RSI', 'Setup A Stocks', 'Volume Analysis'),
            specs=[[{"secondary_y": False}, {"secondary_y": False}],
                   [{"secondary_y": False}, {"secondary_y": False}]]
        )

        # RSI histogram
        fig.add_trace(
            go.Histogram(x=df['rsi'], nbinsx=20, name='RSI Distribution'),
            row=1, col=1
        )

        # Momentum vs RSI scatter
        colors = ['red' if setup else 'blue' for setup in df['setup_a']]
        fig.add_trace(
            go.Scatter(x=df['rsi'], y=df['momentum'], mode='markers',
                      text=df['ticker'], name='Stocks',
                      marker=dict(color=colors)),
            row=1, col=2
        )

        # Setup A stocks
        setup_a_stocks = df[df['setup_a'] == True]
        if len(setup_a_stocks) > 0:
            fig.add_trace(
                go.Bar(x=setup_a_stocks['ticker'], y=setup_a_stocks['momentum'],
                       name='Setup A Momentum'),
                row=2, col=1
            )

        # Volume ratio
        df['volume_ratio'] = df['volume'] / df['volume_avg']
        fig.add_trace(
            go.Scatter(x=df['ticker'], y=df['volume_ratio'], mode='markers',
                      name='Volume Ratio', text=df['ticker']),
            row=2, col=2
        )

        fig.update_layout(height=800, title_text="Technical Analysis Dashboard")
        fig.show()

    def save_results(self, df):
        """Save results and download"""
        if df is None or len(df) == 0:
            return

        # Save to CSV
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"scan_results_{timestamp}.csv"
        df.to_csv(filename, index=False)

        # Create summary report
        setup_a = df[df['setup_a'] == True]
        report = f"""
TECHNICAL SCAN REPORT - {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
=================================================================

SETUP A QUALIFIED: {len(setup_a)} stocks
{setup_a[['ticker', 'price', 'rsi', 'momentum']].to_string(index=False) if len(setup_a) > 0 else 'None'}

HIGH MOMENTUM (>60): {len(df[df['momentum'] > 60])} stocks
TREND CONFIRMED: {len(df[df['trend_confirmed']])} stocks
AVERAGE RSI: {df['rsi'].mean():.1f}

ALERTS:
{chr(10).join(self.alerts) if self.alerts else 'No new alerts'}
        """

        with open(f"report_{timestamp}.txt", 'w') as f:
            f.write(report)

        print(f"📁 Results saved: {filename}")
        print("💾 Download files:")
        files.download(filename)
        files.download(f"report_{timestamp}.txt")

    def start_monitoring(self, duration_hours=8):
        """Start daily monitoring (check multiple times per day)"""
        print(f"🚀 Starting {duration_hours}-hour monitoring with daily analysis")
        print("📈 Professional daily candlestick pattern recognition!")
        print("=" * 70)

        end_time = datetime.now() + timedelta(hours=duration_hours)
        scan_count = 0

        while datetime.now() < end_time:
            clear_output(wait=True)

            print(f"📊 DAILY CANDLESTICK SCANNER - Scan #{scan_count + 1}")
            print(f"📅 Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print(f"🎯 Monitoring until: {end_time.strftime('%H:%M:%S')}")
            print(f"📈 Timeframe: Daily candles for reliable patterns")
            print("=" * 80)

            if self.is_market_hours() or scan_count == 0:  # Always run first scan
                # Run scan
                self.run_scan()
                df = self.display_results()

                # Create visualization every 2nd scan
                if scan_count % 2 == 0 and df is not None:
                    self.create_visualization(df)

                scan_count += 1

                # Save results every 3 scans
                if scan_count % 3 == 0:
                    self.save_results(df)

                print(f"\n⏳ Next scan in 2 hours...")
                print(f"📊 Scans completed: {scan_count}")
                time.sleep(7200)  # Wait 2 hours between scans
            else:
                print("🌙 Market closed - waiting 1 hour...")
                time.sleep(3600)  # Wait 1 hour when market closed

# Initialize scanner
tickers = [ "NVDA","MSFT","AAPL","GOOGL","GOOG","AMZN","META","AVGO","TSLA","NFLX",
    "COST","PLTR","ASML","TMUS","CSCO","AMD","AZN","LIN","PEP","TXN",
    "BKNG","INTU","SHOP","PDD","QCOM","ISRG","AMGN","ADBE","APP","ARM",
    "GILD","HON","MU","AMAT","LRCX","CMCSA","ADI","PANW","ADP","MELI",
    "KLAC","SNPS","INTC","DASH","CRWD","VRTX","SBUX","MSTR","CEG","CDNS",
    "ORLY","CTAS","MDLZ","ABNB","MAR","PYPL","MRVL","CSX","ADSK","MNST",
    "AEP","AXON","NXPI","WDAY","FTNT","REGN","FAST","ROP","PCAR","IDXX",
    "PAYX","ROST","CPRT","EXC","DDOG","TEAM","BKR","EA","XEL","TTWO",
    "KDP","FANG","ZS","CHTR","CCEP","CSGP","VRSK","MCHP","CTSH","GEHC",
    "KHC","ODFL","WBD","DXCM","TTD","LULU","CDW","ON","BIIB","GFS"]

print("🔧 Setting up Colab Stock Scanner...")
scanner = ColabTechnicalScanner(tickers)

print("""
🚀 COLAB DAILY CANDLESTICK SCANNER READY!

📈 PROFESSIONAL DAILY PATTERN RECOGNITION!

USAGE OPTIONS:

1️⃣ SINGLE DAILY SCAN:
   results_df = scanner.run_scan()
   scanner.display_results()

2️⃣ START DAILY MONITORING (8 hours):
   scanner.start_monitoring(duration_hours=8)

3️⃣ FULL ANALYSIS WITH CHARTS:
   results_df = scanner.run_scan()
   df = scanner.display_results()
   scanner.create_visualization(df)

📊 Daily timeframe = More reliable candlestick patterns!
🎯 Original Setup A criteria: RSI 30-40, Volume 1.5x, 2% EMA distance
🔥 10+ Powerful reversal patterns with strength scoring!

Ready for professional daily analysis! 📈
""")

# Uncomment ONE of these to start:
# scanner.start_monitoring(duration_hours=4)  # Monitor for 4 hours with daily analysis
# results_df = scanner.run_scan(); df = scanner.display_results()  # Single daily scan

🔧 Setting up Colab Stock Scanner...

🚀 COLAB DAILY CANDLESTICK SCANNER READY!

📈 PROFESSIONAL DAILY PATTERN RECOGNITION!

USAGE OPTIONS:

1️⃣ SINGLE DAILY SCAN:
   results_df = scanner.run_scan()
   scanner.display_results()

2️⃣ START DAILY MONITORING (8 hours):
   scanner.start_monitoring(duration_hours=8)

3️⃣ FULL ANALYSIS WITH CHARTS:
   results_df = scanner.run_scan()
   df = scanner.display_results()
   scanner.create_visualization(df)

📊 Daily timeframe = More reliable candlestick patterns!
🎯 Original Setup A criteria: RSI 30-40, Volume 1.5x, 2% EMA distance
🔥 10+ Powerful reversal patterns with strength scoring!

Ready for professional daily analysis! 📈



In [ ]:
results_df = scanner.run_scan()
scanner.display_results()

🔍 Starting daily scan at 22:53:54
📊 Analyzing 100 tickers with daily candlestick patterns...
   Progress: 1/100
   Progress: 11/100
   Progress: 21/100
   Progress: 31/100
   Progress: 41/100
   Progress: 51/100
   Progress: 61/100
   Progress: 71/100
   Progress: 81/100
   Progress: 91/100
✅ Daily scan completed in 18.0 seconds
🎯 SETUP A QUALIFIED STOCKS
No stocks currently meet all Setup A criteria
🚀 HIGH MOMENTUM STOCKS (Top 15)

🔥 POWERFUL REVERSAL PATTERNS (2)
🔥 PDD: $128.21 | RSI: 80.97 | Pattern: Bullish Engulfing
🔥 NFLX: $1218.07 | RSI: 68.21 | Pattern: Bullish Engulfing

📊 PATTERN BREAKDOWN:
   Inverted Hammer: 5 stocks
   Hammer: 3 stocks
   Bullish Engulfing: 3 stocks
   Doji (Oversold): 1 stocks

👀 WATCH ZONE STOCKS (1)
⏰ CSGP: $90.51 | RSI: 32.98 | Waiting for signal

📈 SUMMARY
Total analyzed: 100
Setup A qualified: 0
Powerful patterns (3/3): 3
Strong patterns (2/3): 11
Trend confirmed: 36
High momentum (>50): 0
Average RSI: 52.9


,ticker,timestamp,price,ema_21,ema_50,ema_200,rsi,volume,volume_avg,trend_confirmed,...,ema21_distance_pct,bullish_engulfing,morning_star,three_white_soldiers,abandoned_baby,hammer,piercing_pattern,bullish_harami,inverted_hammer,dragonfly_doji
0,MDLZ,2025-08-25 22:53:55,61.96,63.56,65.00,65.07,47.72,5808227,8298741,False,...,-2.52,False,False,False,False,False,False,False,False,False
1,AMD,2025-08-25 22:53:55,163.36,168.23,156.32,133.58,41.26,35941399,69362609,False,...,-2.90,False,False,False,False,False,False,False,False,False
2,AMAT,2025-08-25 22:53:55,161.99,173.78,176.42,171.39,33.41,4146678,8269748,False,...,-6.78,False,False,False,False,True,False,False,False,False
3,FTNT,2025-08-25 22:53:55,77.64,85.02,92.50,96.83,28.80,7259793,11443844,False,...,-8.68,False,False,False,False,False,False,False,False,False
4,TMUS,2025-08-25 22:53:55,251.74,248.53,243.46,240.32,70.47,3856375,3849518,True,...,1.29,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,ARM,2025-08-25 22:54:11,137.78,140.91,142.42,137.32,50.89,1846119,5190940,False,...,-2.22,False,False,False,False,False,False,False,False,False
96,MSFT,2025-08-25 22:54:11,504.26,512.70,500.71,456.67,27.24,21552275,23108158,False,...,-1.65,False,False,False,False,False,False,False,False,False
97,HON,2025-08-25 22:54:12,220.61,220.07,222.01,217.92,51.83,2191716,3251745,True,...,0.24,False,False,False,False,False,False,False,False,False
98,KDP,2025-08-25 22:54:12,31.10,34.09,33.79,33.39,33.23,42775836,9899511,False,...,-8.78,False,False,False,False,False,False,False,False,False


In [ ]:
results_df = scanner.run_scan()
df = scanner.display_results()
scanner.create_visualization(df)

🔍 Starting daily scan at 22:54:38
📊 Analyzing 100 tickers with daily candlestick patterns...
   Progress: 1/100
   Progress: 11/100
   Progress: 21/100
   Progress: 31/100
   Progress: 41/100
   Progress: 51/100
   Progress: 61/100
   Progress: 71/100
   Progress: 81/100
   Progress: 91/100
✅ Daily scan completed in 10.0 seconds
🎯 SETUP A QUALIFIED STOCKS
No stocks currently meet all Setup A criteria
🚀 HIGH MOMENTUM STOCKS (Top 15)

🔥 POWERFUL REVERSAL PATTERNS (2)
🔥 PDD: $128.21 | RSI: 80.97 | Pattern: Bullish Engulfing
🔥 NFLX: $1218.07 | RSI: 68.21 | Pattern: Bullish Engulfing

📊 PATTERN BREAKDOWN:
   Inverted Hammer: 5 stocks
   Hammer: 3 stocks
   Bullish Engulfing: 3 stocks
   Doji (Oversold): 1 stocks

👀 WATCH ZONE STOCKS (1)
⏰ CSGP: $90.51 | RSI: 32.98 | Waiting for signal

📈 SUMMARY
Total analyzed: 100
Setup A qualified: 0
Powerful patterns (3/3): 3
Strong patterns (2/3): 11
Trend confirmed: 36
High momentum (>50): 0
Average RSI: 52.9


In [ ]:
# Filter for strong candlestick patterns (pattern_strength >= 2)
strong_patterns_df = df[df['pattern_strength'] >= 2].copy()

# Sort by absolute distance from EMA 21
strong_patterns_df['abs_ema21_distance_pct'] = abs(strong_patterns_df['ema21_distance_pct'])
closest_to_ema21_df = strong_patterns_df.sort_values(by='abs_ema21_distance_pct')

# Note: "ascending RSI" is a bit ambiguous for a single snapshot.
# I'll assume you mean stocks with RSI below a certain threshold (e.g., < 50)
# that could be considered for a potential upward move.
# If you meant something else, please clarify!

# Filter for RSI below 50 (as a proxy for potential upside)
potential_upside_df = closest_to_ema21_df[closest_to_ema21_df['rsi'] < 50]

print("Stocks with Strong Candlestick Patterns, Closest to 21 EMA, and RSI < 50:")
display(potential_upside_df[['ticker', 'price', 'rsi', 'ema21_distance_pct', 'pattern_strength', 'patterns_found']])

Stocks with Strong Candlestick Patterns, Closest to 21 EMA, and RSI < 50:


,ticker,price,rsi,ema21_distance_pct,pattern_strength,patterns_found
62,AXON,763.52,26.21,-1.18,2,[Inverted Hammer]
90,KLAC,879.55,49.51,-1.75,3,[Bullish Engulfing]
2,AMAT,161.99,33.41,-6.78,2,[Hammer]
